# Lab: Automate Credit Risk Assessment Test Case Generation from Jira Stories using Open-Source LLMs

In this lab you will build a small **agent** that:

1. Connects to a Jira project and fetches a user story by key.
2. Extracts requirement-style information from the story (summary, description, and acceptance criteria).
3. Uses an open-source LLM from Hugging Face to generate structured test cases. 

> **Tip**: Open-source models may not consistently follow formatting instructions. Review generated output before exporting to CSV or JSON.

## 1. Architecture Overview

High-level flow:

1. **Config and Secrets** - Configure the Jira URL, user email, API token, and issue key. Keep secrets outside the code when possible.
2. **Jira Connector (Tool)** - A Python function `fetch_jira_issue(issue_key)` calls the Jira REST API.
3. **Requirement Extractor** - A function pulls the story summary, description, and optional acceptance criteria from the Jira issue JSON.
4. **Open-Source LLM Wrapper** - A Hugging Face text-generation pipeline loads the selected instruction-tuned model.
5. **Test Generation Agent** - The orchestrator fetches the story, extracts requirements, builds a prompt, calls the model, and returns test cases in Markdown.
6. **Export** - The Markdown table is converted to CSV and JSON.

The sample business workflow is **Automate Credit Risk Assessment for New Corporate Customers**. It verifies submitted documents, performs company and sanctions checks, calculates financial risk measures, retrieves external risk data, and recommends credit terms.

## Story details

**Summary**  
Automate Credit Risk Assessment for New Corporate Customers

**Issue Type**  
Story

**Description**

**User Story**  
As a credit analyst, I want the system to assess the credit risk of a new corporate customer using registration documents, financial information, external risk data, and industry risk so that I can assign an appropriate risk category, credit limit, and payment terms.

**Business Context**  
Credit analysts currently review corporate customer documents and financial information manually. This process is time-consuming, inconsistent, and difficult to audit. An automated assessment workflow will validate required evidence, identify compliance concerns, calculate transparent financial measures, combine internal and external risk signals, and provide a consistent recommendation for review.

---

### Functional Requirements

1. **Document Verification**
   - Verify that a Company Registration Certificate has been uploaded.
   - Verify that a GST Certificate has been uploaded.
   - Reject or flag the assessment when a required document is missing, unreadable, or invalid.

2. **Company and Compliance Checks**
   - Match the company name across the registration certificate, GST certificate, and customer record.
   - Verify the company's MCA status.
   - Perform sanctions screening and flag a potential match for analyst review.

3. **Financial Validation and Ratios**
   - Validate that two years of financial statements have been provided and are readable.
   - Calculate Current Ratio as current assets divided by current liabilities.
   - Calculate Debt-to-Equity Ratio as total debt divided by shareholders' equity.
   - Calculate Net Profit Margin as net profit divided by revenue.
   - Do not calculate a ratio when its denominator is zero or the financial input is invalid.

4. **Credit Risk Scorecard**
   - Calculate a final score out of 100 using these component weights:
     - Compliance: 20 points
     - Liquidity: 20 points
     - Leverage: 15 points
     - Profitability: 15 points
     - External Credit Bureau: 15 points
     - Industry Risk: 15 points
   - Use the following Current Ratio scoring thresholds for Liquidity:
     - Current Ratio >= 1.5: 20 points
     - Current Ratio 1.2-1.49: 15 points
     - Current Ratio 1.0-1.19: 8 points
     - Current Ratio < 1.0: 0 points
   - Use the following Debt-to-Equity scoring thresholds for Leverage:
     - Debt-to-Equity <= 0.5: 15 points
     - Debt-to-Equity 0.51-1.0: 12 points
     - Debt-to-Equity 1.01-2.0: 6 points
     - Debt-to-Equity > 2.0: 0 points
   - Use the following Net Profit Margin scoring thresholds for Profitability:
     - Net Profit Margin >= 15%: 15 points
     - Net Profit Margin 10%-14.99%: 12 points
     - Net Profit Margin 5%-9.99%: 6 points
     - Net Profit Margin < 5%: 0 points
   - Assign External Credit Bureau points from the D&B score:
     - D&B score >= 700: 15 points
     - D&B score 650-699: 10 points
     - D&B score 600-649: 5 points
     - D&B score < 600: 0 points
   - Assign Industry Risk points from this matrix:
     - Low Risk, 15 points: Utilities, Government, Healthcare
     - Medium Risk, 8 points: Manufacturing, FMCG Distribution, IT Services
     - High Risk, 3 points: Construction, Real Estate Developers, Airlines, Commodity Trading, Startups, Mining and Metal
   - Assign Compliance points according to the configured compliance policy when required documents, company-name matching, MCA status, and sanctions screening pass.

5. **Risk Classification Rules**
   - Score > 80: Low Risk
   - Score 50-80, inclusive: Medium Risk
   - Score < 50: High Risk
   - Apply the boundaries exactly as written, including scores of 50 and 80.

6. **Assessment and Recommendation**
   - Retrieve the customer's D&B score.
   - Determine the Industry Risk Rating using the industry risk matrix.
   - Calculate the final credit risk score from the configured document, compliance, financial, external, and industry inputs.
   - Assign a risk category based on the final score.
   - Recommend a credit limit and payment terms based on the risk category and assessment rules.
   - Store the assessment inputs, calculations, result, and recommendation for audit purposes.

---

### Non-Functional Requirements

1. A complete assessment must produce a result within 10 seconds when all required services respond normally.
2. The assessment must be deterministic and use the documented scorecard, calculation, threshold, and risk-category rules.
3. Sensitive customer and financial data must be protected in transit and at rest, and must not be written to application logs.
4. The workflow must provide clear validation errors and support analyst review for compliance or data-quality exceptions.
5. The assessment result and supporting calculations must be traceable for audit purposes.

---

### Acceptance Criteria

1. **Required Documents**
   - Given a new corporate customer assessment is started
   - When the Company Registration Certificate and GST Certificate are uploaded and valid
   - Then the assessment may continue to the next validation step.
   - When either required certificate is missing, unreadable, or invalid
   - Then the assessment must be flagged and must not produce a final recommendation.

2. **Company Name and MCA Validation**
   - Given valid registration and GST certificates
   - When the company names match and the MCA status is active
   - Then the company validation must pass.
   - When the names do not match or the MCA status is not active
   - Then the assessment must be flagged for review.

3. **Sanctions Screening**
   - Given a corporate customer is being assessed
   - When sanctions screening returns no potential match
   - Then the compliance check must pass.
   - When screening returns a potential match
   - Then the assessment must be flagged for analyst review and must not automatically approve credit terms.

4. **Financial Statements and Ratio Thresholds**
   - Given two years of valid financial statements
   - When the statements are processed
   - Then the system must calculate Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin using the configured formulas.
   - Then the system must award Liquidity, Leverage, and Profitability points using the documented thresholds.
   - When statements are missing, unreadable, or contain invalid or zero denominators
   - Then the assessment must report a validation error and must not calculate an invalid ratio.

5. **Scorecard Component Boundaries**
   - Given valid assessment inputs
   - When a D&B score is 700, 650, 600, or just below one of those boundaries
   - Then the system must award 15, 10, 5, or 0 points according to the D&B thresholds.
   - When the customer belongs to an industry in the Low, Medium, or High matrix
   - Then the system must award 15, 8, or 3 Industry Risk points.
   - The final score must equal the sum of Compliance, Liquidity, Leverage, Profitability, External Credit Bureau, and Industry Risk points, with a maximum of 100.

6. **Risk Classification and Recommendation**
   - Given a final score above 80, between 50 and 80 inclusive, or below 50
   - Then the system must assign Low Risk, Medium Risk, or High Risk respectively.
   - And it must recommend a credit limit and payment terms consistent with that category.
   - The assessment must retain the inputs, calculations, final score, category, and recommendation for audit.

7. **Response Time and Data Protection**
   - Given normal responses from dependencies
   - When a complete assessment is submitted
   - Then the result must be returned within 10 seconds.
   - Sensitive document contents and financial values must not appear in application logs.

## 2. Environment Setup

Run the installation cell once in a fresh Python 3.12.10 environment. In a managed environment, skip packages that are already installed. The selected model may require additional memory or a GPU.

In [29]:
# If you are using a fresh Python 3.12.10 environment, run this cell once.
%pip install -q transformers torch requests

Note: you may need to restart the kernel to use updated packages.


## 3. Imports


In [30]:
import getpass
import json
import os
from typing import Any, Dict

import requests
import torch
from transformers import pipeline

## 4. Jira Configuration

Read Jira configuration from environment variables to avoid hard-coding secrets.

Required values:

- `JIRA_BASE_URL` - The Jira Cloud URL, for example `https://your-domain.atlassian.net`.
- `JIRA_EMAIL` - The email address associated with your Jira account.
- `JIRA_API_TOKEN` - A Jira API token created at https://id.atlassian.com/manage-profile/security/api-tokens.
- `ISSUE_KEY` - The key of the Jira story that contains the credit-risk requirements, for example `CREDIT-101`.

You can set these values in your shell before starting Jupyter or enter them interactively in the notebook. Interactive values are kept in memory and are not written to a file.

### Obtain Jira values

1. Sign in to Jira Cloud and copy the base URL from the browser address bar. Remove any path after the domain and assign it to `JIRA_BASE_URL`.
2. Use the email address for your Jira account as `JIRA_EMAIL`.
3. Create an API token from the Atlassian account security page and assign it to `JIRA_API_TOKEN`.
4. In Jira, create or open a Story describing **Automate Credit Risk Assessment for New Corporate Customers**. Copy its key, such as `CREDIT-101`, into `ISSUE_KEY`.

Do not commit API tokens or other credentials to source control.

In [ ]:
# --- Jira configuration ---
# Replace the empty values with your Jira details, or set them before starting Jupyter.
os.environ.setdefault("JIRA_BASE_URL", "<replace_with_your_JIRA_base_url>")
os.environ.setdefault("JIRA_EMAIL", "<replace_with_your_mail>")
os.environ.setdefault("JIRA_API_TOKEN", "<replace_with_your_JIRA_API_TOKEN>")
os.environ.setdefault("ISSUE_KEY", "<replace_with_your_issue_key>")

if not os.environ["JIRA_BASE_URL"]:
    os.environ["JIRA_BASE_URL"] = input("Enter your Jira base URL: ").strip().rstrip("/")
if not os.environ["JIRA_EMAIL"]:
    os.environ["JIRA_EMAIL"] = input("Enter your Jira email: ").strip()
if not os.environ["JIRA_API_TOKEN"]:
    os.environ["JIRA_API_TOKEN"] = getpass.getpass("Enter your Jira API token: ")

JIRA_BASE_URL = os.environ["JIRA_BASE_URL"].rstrip("/")
JIRA_EMAIL = os.environ["JIRA_EMAIL"]
JIRA_API_TOKEN = os.environ["JIRA_API_TOKEN"]
ISSUE_KEY = os.environ["ISSUE_KEY"]

## 5. Jira Connector (Tool)

This helper calls the Jira Cloud v3 REST API and returns the issue JSON from `/rest/api/3/issue/{issueKey}`. The issue description contains the credit-risk requirements that will be sent to the open-source model.

In [32]:
def fetch_jira_issue(issue_key: str) -> Dict[str, Any]:
    """Fetch a Jira issue by key and return the JSON response."""
    if not issue_key:
        raise ValueError("Set ISSUE_KEY to a Jira story key before fetching an issue.")

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{issue_key}"
    auth = (JIRA_EMAIL, JIRA_API_TOKEN)
    response = requests.get(url, auth=auth, timeout=30)

    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        print("Error while calling Jira:", error)
        print("Response text:", response.text[:1000])
        raise

    return response.json()


# Optional smoke test. Set ISSUE_KEY before running this cell.
if ISSUE_KEY:
    test_issue = fetch_jira_issue(ISSUE_KEY)
    issue_type = test_issue.get("fields", {}).get("issuetype", {}).get("name")
    print(f"Fetched issue {ISSUE_KEY}. Issue type: {issue_type}")
else:
    print("Set ISSUE_KEY to run the Jira smoke test.")

Fetched issue MDP-8. Issue type: Idea


## 6. Requirement Extractor

Jira fields can contain plain text or rich-text JSON. This lab will:

- Always extract `summary`.
- Convert `description` and other fields to readable strings.
- Preserve the full credit-risk workflow, including functional requirements, non-functional requirements, and acceptance criteria written in the story description.
- Optionally extract acceptance criteria from a custom field when the Jira project stores them separately.

If your Jira project uses a custom acceptance-criteria field, update `AC_FIELD_ID` to its field ID, such as `customfield_10026`.

In [33]:
AC_FIELD_ID = os.environ.get("AC_FIELD_ID", "")


def normalize_field(field_value: Any) -> str:
    """Convert a Jira field, including nested rich text, into a prompt-friendly string."""
    if field_value is None:
        return ""
    if isinstance(field_value, str):
        return field_value
    return json.dumps(field_value, indent=2, ensure_ascii=False)


def extract_requirements(issue_json: Dict[str, Any]) -> Dict[str, str]:
    """Extract the story summary, description, and optional acceptance criteria."""
    fields = issue_json.get("fields", {})
    acceptance_criteria = fields.get(AC_FIELD_ID, "") if AC_FIELD_ID else ""

    return {
        "summary": normalize_field(fields.get("summary", "")),
        "description": normalize_field(fields.get("description", "")),
        "acceptance_criteria": normalize_field(acceptance_criteria),
    }


# Example preview after the Jira connector has fetched an issue.
if "test_issue" in globals():
    reqs_preview = extract_requirements(test_issue)
    print("Summary:\n", reqs_preview["summary"], "\n")
    print("Description (truncated):\n", reqs_preview["description"][:500], "\n")

Summary:
 Automate Credit Risk Assessment for New Corporate Customers - Open Source LLM Test Generation 

Description (truncated):
 {
  "type": "doc",
  "version": 1,
  "content": [
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "User Story"
        }
      ],
      "attrs": {
        "localId": "7e17d289bf00"
      }
    },
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "As a credit analyst, I want the system to assess the credit risk of a new corporate customer using registration documents, financial inform 



## 7. Open-Source LLM Setup

Use a small instruction-tuned model from Hugging Face through `transformers.pipeline`. The default model is TinyLlama because it runs on modest hardware. Output quality may be lower than Azure OpenAI, and larger instruction-tuned models may produce better test cases. Set `MODEL_NAME` to another compatible open-source model when required.

In [34]:
MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "cuda:0" if device == 0 else "cpu")

llm = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
)


def generate_with_llm(prompt: str, max_new_tokens: int = 500) -> str:
    """Generate deterministic test cases with the selected Hugging Face model."""
    outputs = llm(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    generated_text = outputs[0].get("generated_text", "")
    return generated_text.strip()

Using device: cpu


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Users\UdaySomapuram\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\UdaySomapuram\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## 8. Prompt Engineering for Test Generation

The model will act as a senior QA engineer and output test cases in a Markdown table.

Columns:

- `ID`
- `Title`
- `Type (Positive/Negative/Edge)`
- `Pre-conditions`
- `Steps`
- `Expected Result`
- `Mapped Requirement(s)`
- `Priority`

The generated cases should cover document validation, compliance, ratio calculations, D&B scoring, Industry Risk scoring, risk-category assignment, credit-limit recommendations, payment terms, boundary values, and negative scenarios.

Important output rules:

- Generate between 12 and 20 test cases.
- Return only the Markdown table.
- Do not include explanations, introductory text, or conclusions.
- Every data row must contain exactly 8 columns.
- `Type` must be exactly one of `Positive`, `Negative`, or `Edge`.
- Do not use the pipe character `|` inside any cell content. Use commas, semicolons, or words instead so the export parser can preserve each cell.

Example output shape:

| ID | Title | Type (Positive/Negative/Edge) | Pre-conditions | Steps | Expected Result | Mapped Requirement(s) | Priority |
|---|---|---|---|---|---|---|---|
| TC-001 | Validate required certificates | Positive | Valid company record and readable certificates | Upload the Company Registration Certificate and GST Certificate; submit the assessment | Both documents are accepted and the assessment continues | Document Verification; AC-1 | High |
| TC-002 | Reject zero current liabilities | Negative | Financial statements contain zero current liabilities | Submit the statements for assessment | The Current Ratio is not calculated; a validation error is reported | Financial Validation and Ratios; AC-4 | High |
| TC-003 | Review sanctions potential match | Edge | Screening service returns a potential match | Submit the customer for sanctions screening | The assessment is flagged for analyst review and no automatic credit terms are approved | Sanctions Screening; AC-3 | High |

In [35]:
BASE_TESTGEN_PROMPT = """
You are a senior QA engineer.

Read the Jira story requirements below and generate between 12 and 20 test cases covering:
- Company Registration Certificate and GST Certificate validation
- Company-name matching and MCA status
- Sanctions screening and analyst review
- Two years of financial statements
- Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin calculations
- D&B score and Industry Risk Rating
- Final score, risk category, credit limit, and payment terms
- Auditability, response time, data protection, and validation errors
- Positive, negative, edge, and boundary scenarios

Use the scorecard weights and thresholds exactly as supplied. Do not invent scoring logic.
Include tests for ratio bands, D&B scores of 700, 650, and 600, every industry matrix category,
and final scores of 49, 50, 80, and 81.

IMPORTANT OUTPUT RULES:
- Return ONLY the Markdown table.
- Do not include explanations, introductory text, or conclusions.
- Every data row must contain exactly 8 columns.
- The Type value must be exactly one of: Positive, Negative, Edge.
- Do not use the pipe character (|) inside any cell content. Use commas, semicolons, or words instead.

Format the answer as a Markdown table with exactly these columns:
| ID | Title | Type (Positive/Negative/Edge) | Pre-conditions | Steps | Expected Result | Mapped Requirement(s) | Priority |

Number the test cases TC-001, TC-002, and so on.

Jira story requirements:
Summary:
{summary}

Description:
{description}

Acceptance criteria:
{acceptance_criteria}
"""


def build_testcase_prompt(requirements: Dict[str, str]) -> str:
    return BASE_TESTGEN_PROMPT.format(**requirements)

## 9. Putting It Together: The Test Generation Agent

The agent is a simple orchestrator that:

1. Fetches the Jira story.
2. Extracts the summary, description, and optional acceptance criteria.
3. Builds a prompt containing the credit-risk workflow.
4. Calls the open-source Hugging Face model.
5. Returns the generated test cases in Markdown.

In [36]:
class JiraTestGenerationAgent:
    def __init__(self, issue_key: str):
        self.issue_key = issue_key

    def run(self) -> str:
        issue_json = fetch_jira_issue(self.issue_key)
        requirements = extract_requirements(issue_json)
        prompt = build_testcase_prompt(requirements)

        print("Prompt sent to the open-source model (truncated):")
        print(prompt[:1000], "...\n")

        output = generate_with_llm(prompt)
        print("Generated test cases:\n")
        print(output)
        return output

## 10. Run the Agent on a Jira Story

Set `ISSUE_KEY` to the Jira story for **Automate Credit Risk Assessment for New Corporate Customers**, for example `CREDIT-101`, then run the agent cell. The Jira configuration, model setup, and extraction cells must be completed first.

In [37]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [38]:
ISSUE_KEY = os.environ.get("ISSUE_KEY", "MDP-8")

if not ISSUE_KEY:
    ISSUE_KEY = input("Enter the Jira issue key for the credit-risk story: ").strip()

agent = JiraTestGenerationAgent(issue_key=ISSUE_KEY)
testcases_markdown = agent.run()

Prompt sent to the open-source model (truncated):

You are a senior QA engineer.

Read the Jira story requirements below and generate between 12 and 20 test cases covering:
- Company Registration Certificate and GST Certificate validation
- Company-name matching and MCA status
- Sanctions screening and analyst review
- Two years of financial statements
- Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin calculations
- D&B score and Industry Risk Rating
- Final score, risk category, credit limit, and payment terms
- Auditability, response time, data protection, and validation errors
- Positive, negative, edge, and boundary scenarios

Use the scorecard weights and thresholds exactly as supplied. Do not invent scoring logic.
Include tests for ratio bands, D&B scores of 700, 650, and 600, every industry matrix category,
and final scores of 49, 50, 80, and 81.

IMPORTANT OUTPUT RULES:
- Return ONLY the Markdown table.
- Do not include explanations, introductory text, or conclusions

c:\Users\UdaySomapuram\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\UdaySomapuram\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\UdaySomapuram\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top

Generated test cases:

1. Verify that the system correctly validates the uploaded Company Registration Certificate and GST Certificate.
2. Ensure that the system matches company names across all documents without discrepancies.
3. Confirm that the system accurately identifies the MCA status.
4. Perform sanctions screening checks against known lists.
5. Validate the integrity of the two-year financial statements by ensuring they cover the past two fiscal years.
6. Calculate the current ratio, debt-to-equity ratio, and net profit margin based on the provided financial statements.
7. Retrieve the D&B score from the database.
8. Determine the Industry Risk Rating based on the retrieved D&B score.
9. Calculate the final score by combining the ratios and other relevant metrics.
10. Assign the appropriate risk category based on the calculated final score.
11. Recommend a credit limit and payment terms suitable for the assessed risk level.
12. Ensure audit trails exist for each step in the pro

In [39]:
print(testcases_markdown)

1. Verify that the system correctly validates the uploaded Company Registration Certificate and GST Certificate.
2. Ensure that the system matches company names across all documents without discrepancies.
3. Confirm that the system accurately identifies the MCA status.
4. Perform sanctions screening checks against known lists.
5. Validate the integrity of the two-year financial statements by ensuring they cover the past two fiscal years.
6. Calculate the current ratio, debt-to-equity ratio, and net profit margin based on the provided financial statements.
7. Retrieve the D&B score from the database.
8. Determine the Industry Risk Rating based on the retrieved D&B score.
9. Calculate the final score by combining the ratios and other relevant metrics.
10. Assign the appropriate risk category based on the calculated final score.
11. Recommend a credit limit and payment terms suitable for the assessed risk level.
12. Ensure audit trails exist for each step in the process.
13. Maintain a re

In [40]:
import csv
import json

if "testcases_markdown" not in globals() or not testcases_markdown:
    raise ValueError(
        "Run the agent cell first so generated test cases are available to export."
    )

# ---------- 1. Parse Markdown table into rows ----------

lines = [line.strip() for line in testcases_markdown.splitlines()]

table_lines = []

for line in lines:
    if "|" not in line:
        continue

    # Skip Markdown separator rows such as:
    # |---|---|---|
    cells = [cell.strip() for cell in line.strip("|").split("|")]

    if cells and all(
        cell.replace("-", "").replace(":", "").strip() == ""
        for cell in cells
    ):
        continue

    table_lines.append(line)

# Handle models that do not return Markdown tables
if len(table_lines) < 2:
    print("No Markdown table detected in model output.")
    print("The model may not have followed the Markdown table format.")
    print("\nRaw model output preview:\n")
    print(testcases_markdown[:2000])

    rows = []
    headers = []

else:
    # First row is treated as the header
    headers = [
        cell.strip()
        for cell in table_lines[0].strip("|").split("|")
    ]

    rows = []

    # Remaining rows are test cases
    for row_line in table_lines[1:]:
        cells = [
            cell.strip()
            for cell in row_line.strip("|").split("|")
        ]

        # Make sure every row has the same number of columns
        while len(cells) < len(headers):
            cells.append("")

        # Ignore extra cells if the model returned too many
        cells = cells[:len(headers)]

        row_dict = dict(zip(headers, cells))
        rows.append(row_dict)

    print(f"Parsed {len(rows)} test cases")
    print(f"Columns: {headers}")

# ---------- 2. Save CSV ----------

if rows:
    csv_filename = f"{ISSUE_KEY}_credit_risk_testcases.csv"

    with open(
        csv_filename,
        "w",
        newline="",
        encoding="utf-8"
    ) as csvfile:
        writer = csv.DictWriter(
            csvfile,
            fieldnames=headers
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"CSV saved to {csv_filename}")

    # ---------- 3. Save JSON ----------

    json_filename = f"{ISSUE_KEY}_credit_risk_testcases.json"

    with open(
        json_filename,
        "w",
        encoding="utf-8"
    ) as jsonfile:
        json.dump(
            rows,
            jsonfile,
            ensure_ascii=False,
            indent=2
        )

    print(f"JSON saved to {json_filename}")

else:
    print(
        "Skipping CSV and JSON export because no valid Markdown table was generated."
    )

Parsed 3 test cases
Columns: ['ID', 'Title', 'Type (Positive/Negative/Edge)', 'Pre-conditions', 'Steps', 'Expected Result', 'Mapped Requirement(s)']
CSV saved to MDP-8_credit_risk_testcases.csv
JSON saved to MDP-8_credit_risk_testcases.json


## 11. Extensions & Next Steps

Ideas for extending this lab:

- **Code‑Change Driven Tests**  
  Instead of (or in addition to) Jira stories, parse Git diffs and feed changed functions/classes to the LLM to suggest tests.

- **Save Results Back to Jira**  
  Use the Jira REST API to post the generated test cases as a comment or attach them as a file to the story.

- **Stronger Parsing**  
  Parse Jira rich‑text descriptions and structured acceptance criteria fields more carefully instead of simple JSON dumps.

- **Multi‑Agent Setup**  
  Split responsibilities into multiple agents:
  - _Requirements Agent_ – cleans & normalizes story text.
  - _Test Designer Agent_ – proposes tests.
  - _Reviewer Agent_ – checks coverage and suggests missing cases.
